In [38]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [39]:
df = pd.read_csv('dataset/heart_cleveland_upload.csv')

# Replace '?' with NaN
df = df.replace("?", np.nan)

# Convert columns
df["ca"] = pd.to_numeric(df["ca"], errors="coerce")
df["thal"] = pd.to_numeric(df["thal"], errors="coerce")

# Target convert
df["condition"] = df["condition"].apply(lambda x: 1 if x > 0 else 0)

df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,condition
0,69,1,0,160,234,1,2,131,0,0.1,1,1,0,0
1,69,0,0,140,239,0,0,151,0,1.8,0,2,0,0
2,66,0,0,150,226,0,0,114,0,2.6,2,0,0,0
3,65,1,0,138,282,1,2,174,0,1.4,1,1,0,1
4,64,1,0,110,211,0,2,144,1,1.8,1,0,0,0


In [40]:
numeric_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]

categorical_cols = [
    "sex", "cp", "fbs", "restecg",
    "exang", "slope", "ca", "thal"
]

In [41]:
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [42]:
X = df.drop("condition", axis=1)
y = df["condition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [43]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

In [44]:
models = {
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(probability=True),
    "DecisionTree": DecisionTreeClassifier(),
    "RandomForest": RandomForestClassifier()
}

best_score = 0
best_model_name = None
best_pipeline = None

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])
    
    pipeline.fit(X_train, y_train)
    score = pipeline.score(X_test, y_test)
    
    print(f"{name} Accuracy: {score:.4f}")
    
    if score > best_score:
        best_score = score
        best_model_name = name
        best_pipeline = pipeline

KNN Accuracy: 0.7167
SVC Accuracy: 0.7500
DecisionTree Accuracy: 0.7333
RandomForest Accuracy: 0.7500


In [45]:
print(f"\n🏆 Best Model: {best_model_name}")


🏆 Best Model: SVC


In [46]:
y_pred = best_pipeline.predict(X_test)

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Confusion Matrix:
 [[23  9]
 [ 6 22]]

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.72      0.75        32
           1       0.71      0.79      0.75        28

    accuracy                           0.75        60
   macro avg       0.75      0.75      0.75        60
weighted avg       0.75      0.75      0.75        60



In [47]:
joblib.dump(best_pipeline, "heart_pipeline.pkl")

print("✅ Pipeline saved successfully!")

✅ Pipeline saved successfully!


In [48]:
def predict_heart(data: dict):
    model = joblib.load("heart_pipeline.pkl")

    df = pd.DataFrame([data])

    prob = model.predict_proba(df)[0][1] * 100

    if prob >= 70:
        chance = "HIGH"
    elif prob >= 30:
        chance = "MEDIUM"
    else:
        chance = "LOW"

    print(f"🎯 Probability: {prob:.2f}% | Risk: {chance}")
    return prob, chance

In [49]:
test_data = {
    "age": 65, "sex": 1, "cp": 3, "trestbps": 180, "chol": 300,
    "fbs": 1, "restecg": 2, "thalach": 120, "exang": 1,
    "oldpeak": 3.0, "slope": 0, "ca": 3, "thal": 3
}

predict_heart(test_data)

🎯 Probability: 87.20% | Risk: HIGH


(np.float64(87.20464117793405), 'HIGH')